### SQL Databases (Any Structured Data - e.g., SQLite, Oracle, MySQL)

In [1]:
# Create sample SQLite Database
import sqlite3
import os

os.makedirs("data/databases", exist_ok=True)

In [2]:
# Create sample database
conn=sqlite3.connect('data/databases/company.db')
cursor=conn.cursor()  # CURD operations

In [3]:
# Create tables
cursor.execute('''CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    role TEXT,
    department TEXT,
    salary REAL
)''')

In [4]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    status TEXT,
    budget REAL,
    lead_id INTEGER
)''')

In [5]:
# Insert sample data
employees = [
    (1, "John Doe", "Software Engineer", "Engineering", 75000),
    (2, "Jane Smith", "Project Manager", "Management", 85000),
    (3, "Bob Johnson", "Data Analyst", "Analytics", 65000),
    (4, "Alice Brown", "UX Designer", "Design", 70000)
]

projects = [
    (1, "RAG Implementation", "Active", 100000, 1),
    (2, "Data Pipeline", "Completed", 150000, 2),
    (3, "Customer Portal", "Planning", 200000, 3),
    (4, "ML Platform", "Active", 250000, 2)
]

In [8]:
cursor.executemany("INSERT OR REPLACE INTO employees (id, name, role, department, salary) VALUES (?, ?, ?, ?, ?)", employees)
cursor.executemany("INSERT OR REPLACE INTO projects (id, name, status, budget, lead_id) VALUES (?, ?, ?, ?, ?)", projects)

In [9]:
cursor.execute("select * from employees")

In [10]:
conn.commit()
conn.close()

### Database Content Extraction

In [17]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

### Method - SQLDatabase Utility - Reading the DB data

In [24]:
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

# Get Database Info
print(f"Tables: {db.get_usable_table_names()}")
print(f"\nTable DDL:")
for table in db.get_usable_table_names():
    print(f"{table}: {db.get_table_info([table])}")

Tables: ['employees', 'projects']

Table DDL:
employees: 
CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	John Doe	Software Engineer	Engineering	75000.0
2	Jane Smith	Project Manager	Management	85000.0
3	Bob Johnson	Data Analyst	Analytics	65000.0
*/
projects: 
CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	RAG Implementation	Active	100000.0	1
2	Data Pipeline	Completed	150000.0	2
3	Customer Portal	Planning	200000.0	3
*/


### Method - Custom SQL to Document Conversion

In [46]:
from typing import List
from langchain_core.documents import Document

def sql_to_documents(db_path: str) -> List[Document]:
    """Conver SQL Database to Dcouments with context"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    documents = []

    # Strategy 1: Create document for each table
    cursor.execute("SELECT name from sqlite_master where type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]
        # Get table schema
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        # Get table data
        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()

        # Create table overview document
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {', '.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"

        # Add sample records
        table_content += "Sample Records:\n"
        for row in rows[:5]:  # Limit to first 5 records
            record = dict(zip(column_names, row))
            table_content += f"{record}\n"
        doc = Document(
            page_content=table_content,
            metadata={
                'source': db_path,
                'table_name': table_name,
                'num_records': len(rows),
                'data_type': 'sql_table'
            }
        )
        documents.append(doc)
            
    # Strategy 2: Create relationship documents
    # Example: Join employees and projects
    cursor.execute("""SELECT e.name, e.role, p.name as project_name, p.status 
                    FROM employees e JOIN projects p ON e.id = p.lead_id""")
    
    relationships = cursor.fetchall()
    rel_content = "Employee-Project Relationships:\n"
    for rel in relationships:
        rel_content += f"{rel[0]} ({rel[1]}) is leading the project '{rel[2]}' (Status: {rel[3]})\n"
    rel_doc = Document(
        page_content=rel_content,
        metadata={
            'source': db_path,
            'data_type': 'sql_relationships',
            'query': 'employee_project_join'
        }
    )
    documents.append(rel_doc)
    conn.close()
    return documents

In [47]:
sql_to_documents('data/databases/company.db')


[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employees', 'num_records': 4, 'data_type': 'sql_table'}, page_content="Table: employees\nColumns: id, name, role, department, salary\nTotal Records: 4\n\nSample Records:\n{'id': 1, 'name': 'John Doe', 'role': 'Software Engineer', 'department': 'Engineering', 'salary': 75000.0}\n{'id': 2, 'name': 'Jane Smith', 'role': 'Project Manager', 'department': 'Management', 'salary': 85000.0}\n{'id': 3, 'name': 'Bob Johnson', 'role': 'Data Analyst', 'department': 'Analytics', 'salary': 65000.0}\n{'id': 4, 'name': 'Alice Brown', 'role': 'UX Designer', 'department': 'Design', 'salary': 70000.0}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'table_name': 'projects', 'num_records': 4, 'data_type': 'sql_table'}, page_content="Table: projects\nColumns: id, name, status, budget, lead_id\nTotal Records: 4\n\nSample Records:\n{'id': 1, 'name': 'RAG Implementation', 'status': 'Active', 'budget': 100000.0, 'lead_id':